# Notebook 07 — Préparation des données pour le Dashboard

## Objectif

L'objectif de ce notebook est de préparer les datasets finaux
qui seront utilisés par le dashboard destiné au thérapeute.

Nous allons préparer :

- les KPIs globaux ;
- les KPIs par patient ;
- les KPIs par exercice ;
- les KPIs par jeu ;
- les KPIs par niveau ;
- les données de progression ;
- les séances récentes ;
- les données détaillées d'un patient ;
- les données nécessaires aux graphiques du dashboard.

Le dashboard utilisera ensuite ces données pour afficher
des informations simples et utiles au thérapeute.

In [29]:
import pandas as pd
import numpy as np
import os

In [30]:
PROCESSED_PATH = "../data/processed/"

In [31]:
session_analysis = pd.read_csv(
    PROCESSED_PATH + "session_analysis.csv"
)

In [32]:
dashboard_progression = pd.read_csv(
    PROCESSED_PATH + "dashboard_progression.csv"
)

patient_progression_dashboard = pd.read_csv(
    PROCESSED_PATH + "patient_progression_dashboard.csv"
)

exercise_progression = pd.read_csv(
    PROCESSED_PATH + "exercise_progression.csv"
)

game_progression = pd.read_csv(
    PROCESSED_PATH + "game_progression.csv"
)

level_progression = pd.read_csv(
    PROCESSED_PATH + "level_progression.csv"
)

progression_summary = pd.read_csv(
    PROCESSED_PATH + "progression_summary.csv"
)

In [33]:
print("session_analysis :", session_analysis.shape)
print("dashboard_progression :", dashboard_progression.shape)
print("patient_progression_dashboard :", patient_progression_dashboard.shape)
print("exercise_progression :", exercise_progression.shape)
print("game_progression :", game_progression.shape)
print("level_progression :", level_progression.shape)
print("progression_summary :", progression_summary.shape)

session_analysis : (500, 36)
dashboard_progression : (500, 17)
patient_progression_dashboard : (20, 13)
exercise_progression : (8, 8)
game_progression : (3, 6)
level_progression : (3, 7)
progression_summary : (1, 8)


In [34]:
session_analysis["session_date"] = pd.to_datetime(
    session_analysis["session_date"]
)

dashboard_progression["session_date"] = pd.to_datetime(
    dashboard_progression["session_date"]
)

In [35]:
print("Nombre de sessions :", len(session_analysis))
print("Nombre de patients :", session_analysis["patient_id"].nunique())
print("Nombre d'exercices :", session_analysis["exercise_id"].nunique())
print("Nombre de jeux :", session_analysis["game_id"].nunique())

Nombre de sessions : 500
Nombre de patients : 20
Nombre d'exercices : 8
Nombre de jeux : 3


In [36]:
global_kpis = pd.DataFrame({
    "metric": [
        "patients",
        "sessions",
        "exercises",
        "games",
        "mean_score",
        "mean_success_rate",
        "mean_duration",
        "mean_repetitions",
        "mean_progression"
    ],
    "value": [
        session_analysis["patient_id"].nunique(),
        len(session_analysis),
        session_analysis["exercise_id"].nunique(),
        session_analysis["game_id"].nunique(),
        session_analysis["score"].mean(),
        session_analysis["success_rate"].mean(),
        session_analysis["duration"].mean(),
        session_analysis["repetitions"].mean(),
        session_analysis["progression"].mean()
    ]
})

In [37]:
global_kpis["value"] = global_kpis["value"].round(2)

display(global_kpis)

,metric,value
0,patients,20.00
1,sessions,500.00
2,exercises,8.00
3,games,3.00
4,mean_score,56.78
5,mean_success_rate,57.34
6,mean_duration,7.84
7,mean_repetitions,14.16
8,mean_progression,3.00


In [38]:
improved_sessions = (
    session_analysis["progression"] > 0
).sum()

declined_sessions = (
    session_analysis["progression"] < 0
).sum()

global_evolution_kpis = pd.DataFrame({
    "metric": [
        "improved_sessions",
        "declined_sessions",
        "sessions_without_progression"
    ],
    "value": [
        improved_sessions,
        declined_sessions,
        session_analysis["progression"].isna().sum()
    ]
})

display(global_evolution_kpis)

,metric,value
0,improved_sessions,260
1,declined_sessions,196
2,sessions_without_progression,43


In [39]:
global_kpis_final = pd.concat(
    [
        global_kpis,
        global_evolution_kpis
    ],
    ignore_index=True
)

display(global_kpis_final)

,metric,value
0,patients,20.00
1,sessions,500.00
2,exercises,8.00
3,games,3.00
4,mean_score,56.78
5,mean_success_rate,57.34
6,mean_duration,7.84
7,mean_repetitions,14.16
8,mean_progression,3.00
9,improved_sessions,260.00


In [40]:
patient_kpis = (
    session_analysis
    .groupby(
        ["patient_id", "first_name", "last_name"]
    )
    .agg(
        sessions=("id_session", "count"),
        mean_score=("score", "mean"),
        mean_success_rate=("success_rate", "mean"),
        mean_duration=("duration", "mean"),
        mean_repetitions=("repetitions", "mean"),
        mean_progression=("progression", "mean")
    )
    .reset_index()
)

In [41]:
patient_kpis = patient_kpis.merge(
    patient_progression_dashboard[
        [
            "patient_id",
            "sessions_with_progression",
            "max_progression",
            "min_progression",
            "improved_sessions",
            "declined_sessions",
            "improvement_rate",
            "progression_category"
        ]
    ],
    on="patient_id",
    how="left"
)

In [42]:
numeric_columns = [
    "mean_score",
    "mean_success_rate",
    "mean_duration",
    "mean_repetitions",
    "mean_progression",
    "max_progression",
    "min_progression",
    "improvement_rate"
]

patient_kpis[numeric_columns] = (
    patient_kpis[numeric_columns].round(2)
)

In [43]:
print("Nombre de patients :", len(patient_kpis))

display(
    patient_kpis.head(10)
)

Nombre de patients : 20


,patient_id,first_name,last_name,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression,sessions_with_progression,max_progression,min_progression,improved_sessions,declined_sessions,improvement_rate,progression_category
0,1,Yasmine,Bouzid,26,60.97,60.91,6.62,10.50,3.24,24,21.52,-14.53,14,10,58.33,Progression modérée
1,2,Amira,Jebali,24,54.94,56.56,7.17,12.33,1.34,22,28.46,-23.37,11,11,50.00,Progression faible
2,3,Fatma,Tlili,32,56.43,59.27,9.22,17.84,1.79,29,32.78,-25.09,16,13,55.17,Progression faible
3,4,Malek,Hamdi,26,55.06,55.61,8.73,17.04,1.84,24,37.40,-15.31,11,13,45.83,Progression faible
4,5,Zied,Gharbi,42,59.17,60.23,7.69,13.57,4.32,39,29.74,-18.71,26,13,66.67,Progression modérée
5,6,Eya,Haddad,66,59.53,59.47,7.73,14.89,3.06,62,30.53,-20.93,36,26,58.06,Progression modérée
6,7,Yassine,Sassi,31,57.83,58.08,8.03,13.61,3.48,29,27.73,-14.90,16,13,55.17,Progression modérée
7,8,Hiba,Khelifi,13,64.07,64.32,6.46,11.69,2.63,12,14.08,-9.06,8,4,66.67,Progression modérée
8,9,Adam,Mejri,25,55.15,55.92,6.08,9.40,2.56,23,17.69,-10.73,12,10,52.17,Progression modérée
9,10,Maya,Trabelsi,30,61.27,59.21,6.60,10.67,3.00,28,28.02,-16.71,18,10,64.29,Progression modérée


In [44]:
exercise_kpis = (
    session_analysis
    .groupby(["exercise_id", "name"])
    .agg(
        sessions=("id_session", "count"),
        mean_score=("score", "mean"),
        mean_success_rate=("success_rate", "mean"),
        mean_duration=("duration", "mean"),
        mean_repetitions=("repetitions", "mean"),
        mean_progression=("progression", "mean")
    )
    .reset_index()
)

In [45]:
exercise_numeric_columns = [
    "mean_score",
    "mean_success_rate",
    "mean_duration",
    "mean_repetitions",
    "mean_progression"
]

exercise_kpis[exercise_numeric_columns] = (
    exercise_kpis[exercise_numeric_columns].round(2)
)

display(exercise_kpis)

,exercise_id,name,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression
0,1,Forward Arm Raise,57,51.75,52.49,7.84,14.14,2.36
1,2,Lateral Arm Raise,30,60.43,61.29,8.20,14.50,3.23
2,3,Overhead Reach,17,50.45,53.78,7.88,14.94,1.35
3,4,Seated Knee Extension,56,59.24,59.92,7.38,13.14,3.79
4,5,Mini Squat,62,56.38,56.65,7.52,13.85,3.14
5,6,Step Up,79,58.37,58.65,7.73,13.85,2.16
6,7,Wrist Flexion and Extension,92,58.37,57.71,8.20,14.57,2.11
7,8,Forearm Rotation,107,55.84,57.15,7.94,14.56,4.45


In [46]:
game_kpis = (
    session_analysis
    .groupby(["game_id", "name_game"])
    .agg(
        sessions=("id_session", "count"),
        mean_score=("score", "mean"),
        mean_success_rate=("success_rate", "mean"),
        mean_duration=("duration", "mean"),
        mean_repetitions=("repetitions", "mean"),
        mean_progression=("progression", "mean")
    )
    .reset_index()
)

In [47]:
game_numeric_columns = [
    "mean_score",
    "mean_success_rate",
    "mean_duration",
    "mean_repetitions",
    "mean_progression"
]

game_kpis[game_numeric_columns] = (
    game_kpis[game_numeric_columns].round(2)
)

display(game_kpis)

,game_id,name_game,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression
0,1,Bubble Reach,104,54.04,55.24,7.95,14.38,2.46
1,2,Step Quest,197,57.99,58.38,7.56,13.65,2.93
2,3,Wrist Wizard,199,57.01,57.41,8.06,14.56,3.36


In [48]:
level_kpis = (
    session_analysis
    .groupby("level_number")
    .agg(
        sessions=("id_session", "count"),
        mean_score=("score", "mean"),
        mean_success_rate=("success_rate", "mean"),
        mean_duration=("duration", "mean"),
        mean_repetitions=("repetitions", "mean"),
        mean_progression=("progression", "mean")
    )
    .reset_index()
)

In [49]:
level_numeric_columns = [
    "mean_score",
    "mean_success_rate",
    "mean_duration",
    "mean_repetitions",
    "mean_progression"
]

level_kpis[level_numeric_columns] = (
    level_kpis[level_numeric_columns].round(2)
)

display(level_kpis)

,level_number,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression
0,1,65,51.98,53.45,8.49,16.11,1.72
1,2,248,56.05,56.82,7.92,14.29,3.52
2,3,187,59.41,59.38,7.52,13.32,2.76


In [50]:
recent_sessions = (
    session_analysis
    .sort_values(
        "session_date",
        ascending=False
    )
    .head(20)
    .copy()
)

In [51]:
recent_sessions = recent_sessions[
    [
        "id_session",
        "patient_id",
        "first_name",
        "last_name",
        "name",
        "name_game",
        "level_number",
        "session_date",
        "score",
        "success_rate",
        "repetitions",
        "duration",
        "progression"
    ]
]

In [52]:
display(recent_sessions)

,id_session,patient_id,first_name,last_name,name,name_game,level_number,session_date,score,success_rate,repetitions,duration,progression
475,500,16,Rayan,Mansouri,Mini Squat,Step Quest,3,2026-09-22 16:30:42,46.0,51.7,15,7,-14.50
381,499,5,Zied,Gharbi,Forward Arm Raise,Bubble Reach,3,2026-09-21 15:40:59,59.4,63.5,11,8,-2.14
264,498,6,Eya,Haddad,Forward Arm Raise,Bubble Reach,3,2026-09-19 11:45:07,61.6,70.6,15,10,3.53
288,497,5,Zied,Gharbi,Mini Squat,Step Quest,3,2026-09-19 11:00:59,73.9,69.6,12,8,0.27
171,496,8,Hiba,Khelifi,Forearm Rotation,Wrist Wizard,2,2026-09-19 10:45:56,72.5,65.3,12,6,3.57
421,495,12,Rim,Ayari,Forward Arm Raise,Bubble Reach,1,2026-09-19 10:45:39,48.0,45.8,18,10,17.65
202,494,6,Eya,Haddad,Step Up,Step Quest,3,2026-09-19 10:45:27,76.0,80.2,15,7,6.15
329,493,6,Eya,Haddad,Mini Squat,Step Quest,2,2026-09-19 10:15:43,71.6,68.2,18,10,25.61
403,492,5,Zied,Gharbi,Lateral Arm Raise,Bubble Reach,3,2026-09-19 10:00:58,73.0,71.5,14,8,2.24
227,491,6,Eya,Haddad,Lateral Arm Raise,Bubble Reach,3,2026-09-19 09:30:52,70.7,72.8,15,6,-1.94


In [53]:
def classify_progression(value):
    if pd.isna(value):
        return "Première séance"
    elif value > 0:
        return "Amélioration"
    elif value < 0:
        return "Diminution"
    else:
        return "Stable"


session_analysis["progression_status"] = (
    session_analysis["progression"]
    .apply(classify_progression)
)

In [54]:
print(
    session_analysis["progression_status"].value_counts()
)

progression_status
Amélioration       260
Diminution         196
Première séance     43
Stable               1
Name: count, dtype: int64


In [55]:
progression_history = session_analysis[
    [
        "id_session",
        "patient_id",
        "first_name",
        "last_name",
        "exercise_id",
        "name",
        "game_id",
        "name_game",
        "level_number",
        "session_date",
        "score",
        "success_rate",
        "repetitions",
        "duration",
        "progression",
        "progression_status"
    ]
].copy()

In [56]:
progression_history = progression_history.sort_values(
    [
        "patient_id",
        "session_date"
    ]
).reset_index(drop=True)

In [57]:
display(
    progression_history.head(20)
)

,id_session,patient_id,first_name,last_name,exercise_id,name,game_id,name_game,level_number,session_date,score,success_rate,repetitions,duration,progression,progression_status
0,1,1,Yasmine,Bouzid,7,Wrist Flexion and Extension,3,Wrist Wizard,3,2026-04-30 15:50:31,60.6,60.2,9,4,NaN,Première séance
1,2,1,Yasmine,Bouzid,7,Wrist Flexion and Extension,3,Wrist Wizard,3,2026-05-04 15:00:15,52.5,52.7,9,8,-13.37,Diminution
2,3,1,Yasmine,Bouzid,8,Forearm Rotation,3,Wrist Wizard,2,2026-05-05 15:20:38,47.1,52.7,10,6,NaN,Première séance
3,4,1,Yasmine,Bouzid,7,Wrist Flexion and Extension,3,Wrist Wizard,3,2026-05-07 15:00:09,63.8,68.7,10,5,21.52,Amélioration
4,5,1,Yasmine,Bouzid,8,Forearm Rotation,3,Wrist Wizard,2,2026-05-08 15:20:48,48.7,61.3,13,7,3.40,Amélioration
5,6,1,Yasmine,Bouzid,7,Wrist Flexion and Extension,3,Wrist Wizard,3,2026-05-12 15:10:22,60.3,59.0,6,4,-5.49,Diminution
6,7,1,Yasmine,Bouzid,8,Forearm Rotation,3,Wrist Wizard,2,2026-05-12 15:55:04,57.5,62.4,7,6,18.07,Amélioration
7,8,1,Yasmine,Bouzid,7,Wrist Flexion and Extension,3,Wrist Wizard,3,2026-05-16 10:45:20,62.7,70.3,11,9,3.98,Amélioration
8,9,1,Yasmine,Bouzid,8,Forearm Rotation,3,Wrist Wizard,2,2026-05-18 15:00:36,52.9,55.8,9,6,-8.00,Diminution
9,10,1,Yasmine,Bouzid,7,Wrist Flexion and Extension,3,Wrist Wizard,3,2026-05-21 15:40:50,61.0,60.4,14,7,-2.71,Diminution


In [58]:
def get_patient_history(patient_id):
    
    patient_history = (
        session_analysis[
            session_analysis["patient_id"] == patient_id
        ]
        .sort_values("session_date")
        .copy()
    )
    
    return patient_history

In [59]:
patient_id_test = 1

patient_history = get_patient_history(
    patient_id_test
)

display(
    patient_history[
        [
            "session_date",
            "name",
            "level_number",
            "score",
            "success_rate",
            "repetitions",
            "progression"
        ]
    ]
)

,session_date,name,level_number,score,success_rate,repetitions,progression
0,2026-04-30 15:50:31,Wrist Flexion and Extension,3,60.6,60.2,9,NaN
1,2026-05-04 15:00:15,Wrist Flexion and Extension,3,52.5,52.7,9,-13.37
13,2026-05-05 15:20:38,Forearm Rotation,2,47.1,52.7,10,NaN
2,2026-05-07 15:00:09,Wrist Flexion and Extension,3,63.8,68.7,10,21.52
14,2026-05-08 15:20:48,Forearm Rotation,2,48.7,61.3,13,3.40
3,2026-05-12 15:10:22,Wrist Flexion and Extension,3,60.3,59.0,6,-5.49
15,2026-05-12 15:55:04,Forearm Rotation,2,57.5,62.4,7,18.07
4,2026-05-16 10:45:20,Wrist Flexion and Extension,3,62.7,70.3,11,3.98
16,2026-05-18 15:00:36,Forearm Rotation,2,52.9,55.8,9,-8.00
5,2026-05-21 15:40:50,Wrist Flexion and Extension,3,61.0,60.4,14,-2.71


In [60]:
def get_patient_kpis(patient_id):
    
    patient = patient_kpis[
        patient_kpis["patient_id"] == patient_id
    ]
    
    return patient

In [61]:
display(
    get_patient_kpis(1)
)

,patient_id,first_name,last_name,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression,sessions_with_progression,max_progression,min_progression,improved_sessions,declined_sessions,improvement_rate,progression_category
0,1,Yasmine,Bouzid,26,60.97,60.91,6.62,10.5,3.24,24,21.52,-14.53,14,10,58.33,Progression modérée


In [62]:
last_session_per_patient = (
    session_analysis
    .sort_values("session_date")
    .groupby(
        ["patient_id", "first_name", "last_name"],
        as_index=False
    )
    .tail(1)
)

In [63]:
last_session_per_patient = last_session_per_patient[
    [
        "patient_id",
        "first_name",
        "last_name",
        "session_date",
        "name",
        "name_game",
        "level_number",
        "score",
        "success_rate",
        "progression"
    ]
].copy()

In [64]:
print(
    "Nombre de patients :",
    len(last_session_per_patient)
)

display(last_session_per_patient)

Nombre de patients : 20


,patient_id,first_name,last_name,session_date,name,name_game,level_number,score,success_rate,progression
101,11,Sirine,Ben Salah,2026-07-17 16:40:19,Step Up,Step Quest,2,61.3,53.5,-7.82
12,1,Yasmine,Bouzid,2026-07-21 15:30:00,Wrist Flexion and Extension,Wrist Wizard,3,79.5,67.5,10.26
63,17,Skander,Karoui,2026-07-23 17:20:30,Overhead Reach,Bubble Reach,1,52.7,55.9,20.87
237,2,Amira,Jebali,2026-08-29 10:15:09,Wrist Flexion and Extension,Wrist Wizard,2,57.6,55.6,-0.35
77,9,Adam,Mejri,2026-08-29 10:30:26,Step Up,Step Quest,3,55.7,50.2,-2.96
128,7,Yassine,Sassi,2026-09-05 10:15:34,Wrist Flexion and Extension,Wrist Wizard,2,62.5,61.7,5.75
301,4,Malek,Hamdi,2026-09-05 10:30:26,Wrist Flexion and Extension,Wrist Wizard,2,51.2,59.8,-13.07
390,20,Omar,Dridi,2026-09-08 16:20:03,Overhead Reach,Bubble Reach,1,52.6,53.1,-0.94
457,18,Lina,Chaouch,2026-09-08 16:20:18,Wrist Flexion and Extension,Wrist Wizard,2,57.2,56.5,16.26
354,14,Ilyes,Belhaj,2026-09-09 15:20:18,Wrist Flexion and Extension,Wrist Wizard,3,53.6,46.7,-18.54


In [65]:
score_evolution = session_analysis[
    [
        "patient_id",
        "first_name",
        "last_name",
        "session_date",
        "score"
    ]
].copy()

score_evolution = score_evolution.sort_values(
    ["patient_id", "session_date"]
)

In [66]:
success_evolution = session_analysis[
    [
        "patient_id",
        "first_name",
        "last_name",
        "session_date",
        "success_rate"
    ]
].copy()

success_evolution = success_evolution.sort_values(
    ["patient_id", "session_date"]
)

In [67]:
progression_evolution = session_analysis[
    [
        "patient_id",
        "first_name",
        "last_name",
        "session_date",
        "progression"
    ]
].copy()

progression_evolution = progression_evolution.sort_values(
    ["patient_id", "session_date"]
)

In [68]:
dashboard_summary = {
    "total_patients": int(
        session_analysis["patient_id"].nunique()
    ),
    
    "total_sessions": int(
        len(session_analysis)
    ),
    
    "total_exercises": int(
        session_analysis["exercise_id"].nunique()
    ),
    
    "total_games": int(
        session_analysis["game_id"].nunique()
    ),
    
    "average_score": round(
        session_analysis["score"].mean(), 2
    ),
    
    "average_success_rate": round(
        session_analysis["success_rate"].mean(), 2
    ),
    
    "average_duration": round(
        session_analysis["duration"].mean(), 2
    ),
    
    "average_repetitions": round(
        session_analysis["repetitions"].mean(), 2
    ),
    
    "average_progression": round(
        session_analysis["progression"].mean(), 2
    )
}

dashboard_summary

{'total_patients': 20,
 'total_sessions': 500,
 'total_exercises': 8,
 'total_games': 3,
 'average_score': np.float64(56.78),
 'average_success_rate': np.float64(57.34),
 'average_duration': np.float64(7.84),
 'average_repetitions': np.float64(14.16),
 'average_progression': np.float64(3.0)}

In [69]:
dashboard_summary_df = pd.DataFrame(
    [dashboard_summary]
)

display(dashboard_summary_df)

,total_patients,total_sessions,total_exercises,total_games,average_score,average_success_rate,average_duration,average_repetitions,average_progression
0,20,500,8,3,56.78,57.34,7.84,14.16,3.0


In [70]:
print("Missing values — patient_kpis")
display(patient_kpis.isna().sum())

print("\nMissing values — exercise_kpis")
display(exercise_kpis.isna().sum())

print("\nMissing values — game_kpis")
display(game_kpis.isna().sum())

print("\nMissing values — level_kpis")
display(level_kpis.isna().sum())

Missing values — patient_kpis


patient_id                   0
first_name                   0
last_name                    0
sessions                     0
mean_score                   0
mean_success_rate            0
mean_duration                0
mean_repetitions             0
mean_progression             0
sessions_with_progression    0
max_progression              0
min_progression              0
improved_sessions            0
declined_sessions            0
improvement_rate             0
progression_category         0
dtype: int64


Missing values — exercise_kpis


exercise_id          0
name                 0
sessions             0
mean_score           0
mean_success_rate    0
mean_duration        0
mean_repetitions     0
mean_progression     0
dtype: int64


Missing values — game_kpis


game_id              0
name_game            0
sessions             0
mean_score           0
mean_success_rate    0
mean_duration        0
mean_repetitions     0
mean_progression     0
dtype: int64


Missing values — level_kpis


level_number         0
sessions             0
mean_score           0
mean_success_rate    0
mean_duration        0
mean_repetitions     0
mean_progression     0
dtype: int64

In [71]:
print("===== DIMENSIONS =====")

print("Global KPIs :", global_kpis_final.shape)
print("Patient KPIs :", patient_kpis.shape)
print("Exercise KPIs :", exercise_kpis.shape)
print("Game KPIs :", game_kpis.shape)
print("Level KPIs :", level_kpis.shape)
print("Recent sessions :", recent_sessions.shape)
print("Progression history :", progression_history.shape)
print("Score evolution :", score_evolution.shape)
print("Success evolution :", success_evolution.shape)
print("Progression evolution :", progression_evolution.shape)

===== DIMENSIONS =====
Global KPIs : (12, 2)
Patient KPIs : (20, 16)
Exercise KPIs : (8, 8)
Game KPIs : (3, 8)
Level KPIs : (3, 7)
Recent sessions : (20, 13)
Progression history : (500, 16)
Score evolution : (500, 5)
Success evolution : (500, 5)
Progression evolution : (500, 5)


In [72]:
assert len(patient_kpis) == 20
assert len(exercise_kpis) == 8
assert len(game_kpis) == 3
assert len(level_kpis) == 3

assert len(progression_history) == 500

print("Toutes les vérifications sont passées.")

Toutes les vérifications sont passées.


In [73]:
global_kpis_final.to_csv(
    PROCESSED_PATH + "dashboard_global_kpis.csv",
    index=False
)

In [74]:
patient_kpis.to_csv(
    PROCESSED_PATH + "dashboard_patient_kpis.csv",
    index=False
)

In [75]:
exercise_kpis.to_csv(
    PROCESSED_PATH + "dashboard_exercise_kpis.csv",
    index=False
)

In [76]:
game_kpis.to_csv(
    PROCESSED_PATH + "dashboard_game_kpis.csv",
    index=False
)

In [77]:
level_kpis.to_csv(
    PROCESSED_PATH + "dashboard_level_kpis.csv",
    index=False
)

In [78]:
level_kpis.to_csv(
    PROCESSED_PATH + "dashboard_level_kpis.csv",
    index=False
)

In [79]:
progression_history.to_csv(
    PROCESSED_PATH + "dashboard_progression_history.csv",
    index=False
)

In [80]:
score_evolution.to_csv(
    PROCESSED_PATH + "dashboard_score_evolution.csv",
    index=False
)

In [81]:
success_evolution.to_csv(
    PROCESSED_PATH + "dashboard_success_evolution.csv",
    index=False
)

In [82]:
success_evolution.to_csv(
    PROCESSED_PATH + "dashboard_success_evolution.csv",
    index=False
)

In [83]:
last_session_per_patient.to_csv(
    PROCESSED_PATH + "dashboard_last_session.csv",
    index=False
)

In [84]:
dashboard_summary_df.to_csv(
    PROCESSED_PATH + "dashboard_summary.csv",
    index=False
)

In [85]:
dashboard_files = [
    "dashboard_global_kpis.csv",
    "dashboard_patient_kpis.csv",
    "dashboard_exercise_kpis.csv",
    "dashboard_game_kpis.csv",
    "dashboard_level_kpis.csv",
    "dashboard_recent_sessions.csv",
    "dashboard_progression_history.csv",
    "dashboard_score_evolution.csv",
    "dashboard_success_evolution.csv",
    "dashboard_progression_evolution.csv",
    "dashboard_last_session.csv",
    "dashboard_summary.csv"
]

for file in dashboard_files:
    
    path = PROCESSED_PATH + file
    
    print(
        f"{file} →",
        "OK" if os.path.exists(path) else "MANQUANT"
    )

dashboard_global_kpis.csv → OK
dashboard_patient_kpis.csv → OK
dashboard_exercise_kpis.csv → OK
dashboard_game_kpis.csv → OK
dashboard_level_kpis.csv → OK
dashboard_recent_sessions.csv → MANQUANT
dashboard_progression_history.csv → OK
dashboard_score_evolution.csv → OK
dashboard_success_evolution.csv → OK
dashboard_progression_evolution.csv → MANQUANT
dashboard_last_session.csv → OK
dashboard_summary.csv → OK


In [ ]:
print("====================================")
print("     VALIDATION NOTEBOOK 07")
print("====================================")

print(
    "Patients :",
    session_analysis["patient_id"].nunique()
)

print(
    "Sessions :",
    len(session_analysis)
)

print(
    "Exercices :",
    session_analysis["exercise_id"].nunique()
)

print(
    "Jeux :",
    session_analysis["game_id"].nunique()
)

print(
    "Patient KPI rows :",
    len(patient_kpis)
)

print(
    "Exercise KPI rows :",
    len(exercise_kpis)
)

print(
    "Game KPI rows :",
    len(game_kpis)
)

print(
    "Level KPI rows :",
    len(level_kpis)
)

print(
    "Dashboard files :",
    len(dashboard_files)
)

print("====================================")
print("Notebook 07 terminé.")
print("====================================")

     VALIDATION NOTEBOOK 07
Patients : 20
Sessions : 500
Exercices : 8
Jeux : 3
Patient KPI rows : 20
Exercise KPI rows : 8
Game KPI rows : 3
Level KPI rows : 3
Dashboard files : 12
Notebook 07 terminé.


: 

# Conclusion

Le Notebook 07 a permis de transformer les résultats des analyses
précédentes en datasets directement exploitables par le dashboard.

Les données préparées comprennent :

- les KPIs globaux ;
- les KPIs par patient ;
- les KPIs par exercice ;
- les KPIs par jeu ;
- les KPIs par niveau ;
- les séances récentes ;
- l'historique de progression ;
- l'évolution du score ;
- l'évolution du taux de réussite ;
- l'évolution de la progression ;
- la dernière séance de chaque patient.

Ces datasets constituent la couche analytique entre
les données de rééducation et l'interface du dashboard.

Le prochain objectif est de construire le dashboard thérapeute
à partir de ces datasets.